# Gemma 4 AML Fine-Tune -- GGUF -- HuggingFace Hub

**Pipeline:**
1. Install Unsloth + deps
2. QLoRA fine-tune on `framl_train_combined_v26.jsonl` (713 examples)
3. Merge LoRA -> export F16 GGUF -> quantise Q4_K_M
4. Upload GGUF to HuggingFace Hub
5. Register with Ollama as `aria-v2`

> **NOTE:** Gemma 4 model IDs and chat template tokens should be verified
> against the Unsloth release notes before running -- marked with `# VERIFY` comments.


## Cell 1 -- Install Dependencies
Unsloth must be installed before transformers/trl.

In [ ]:
!/venv/main/bin/pip install unsloth unsloth_zoo transformers trl peft datasets \
    bitsandbytes accelerate sentence-transformers huggingface_hub

# torchcodec pulls in FFmpeg 4.x dependency that conflicts with vast.ai system FFmpeg 5+
!/venv/main/bin/pip uninstall -y torchcodec 2>/dev/null || true

# Flash Attention 2 -- try precompiled wheel only (--only-binary=:all: makes pip fail
# instantly if no wheel exists rather than hanging on a 20-min source build).
# Unsloth has its own built-in attention kernels so training works fine without it.
import subprocess
fa_result = subprocess.run(
    ["/venv/main/bin/pip", "install", "flash-attn",
     "--only-binary=:all:", "--no-deps", "--quiet"],
    capture_output=True, text=True
)
if fa_result.returncode == 0:
    print("flash-attn: installed")
else:
    print("flash-attn: no precompiled wheel for this torch/CUDA version -- skipping")
    print("  Unsloth built-in kernels are active, training unaffected.")

import unsloth; print("Unsloth:", unsloth.__version__)

## Cell 2 -- Environment & Imports
`HF_HOME` must be set **before** any HuggingFace import.

In [ ]:
import os
os.environ['HF_HOME'] = '/dev/shm/huggingface'   # RAM-backed tmpfs -- fast, ~15GB free, wiped on stop
os.environ['HF_TOKEN'] = ''  # paste your token here

import torch
from pathlib import Path
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3 -- Configuration

**GPU guide:**
- RTX 3090 / 4090 (24 GB) -> `gemma-4-4b-it` (E4B), batch=2, accum=4
- A100 40 GB -> `gemma-4-27b-it`, batch=1, accum=8


In [ ]:
# -- Model --
BASE_MODEL  = 'unsloth/gemma-4-E4B-it'   # VERIFY: check Unsloth hub for exact ID
MAX_SEQ_LEN = 4096

# -- LoRA --
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'gate_proj', 'up_proj', 'down_proj']

# -- Training --
EPOCHS     = 3        # 843 examples x 3 epochs
BATCH_SIZE = 2
GRAD_ACCUM = 4        # effective batch = 8
LR         = 2e-4

# -- Paths --
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'finetune':
    REPO_ROOT = REPO_ROOT.parent
DATA_FILE = REPO_ROOT / 'finetune' / 'data' / 'aria_train_combined_v36_full.jsonl'
OUT_DIR   = Path('/workspace/aria-v6-adapter')   # persistent across reboots
GGUF_DIR  = Path('/workspace/aria-v6-gguf')
Q8_GGUF   = Path('/workspace/aria-v6-q8.gguf')
OUT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)

# -- HuggingFace Hub --
HF_REPO  = 'speri420/aria-v2'
HF_TOKEN = os.environ.get('HF_TOKEN', '')

# -- Ollama --
OLLAMA_MODEL_NAME = 'aria-v6'

print(f'Base model: {BASE_MODEL}')
print(f'Data:       {DATA_FILE}  (exists={DATA_FILE.exists()})')
print(f'GGUF out:   {Q8_GGUF}')
print(f'HF repo:    {HF_REPO}  (token set={bool(HF_TOKEN)})')

## Cell 4 -- Load Base Model (4-bit QLoRA)

In [ ]:
print(f'Loading {BASE_MODEL} in 4-bit QLoRA ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print('Model loaded.')
tmpl = tokenizer.chat_template or 'NOT SET'
print(f'Chat template (first 120): {tmpl[:120]}')


## Cell 5 -- Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()


## Cell 6 -- Load & Format Dataset

Gemma 4 uses `<start_of_turn>` / `<end_of_turn>` tokens.
`tokenizer.apply_chat_template()` handles the format automatically.

> **VERIFY:** If tool_calls turns cause errors, the Unsloth Gemma 4 tokenizer
> may need tool role handling -- check Unsloth release notes.


In [ ]:
import json

def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def format_example(example):
    msgs = example['messages']
    # Pass messages through unchanged -- apply_chat_template handles tool_calls
    # and role:tool natively, producing the correct Gemma 4 format:
    #   <|tool_call>call:name{"arg": "val"}<tool_call|>   (tool call)
    #   <|turn>tool
{content}<turn|>                      (tool result)
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {'text': text}

raw = load_jsonl(str(DATA_FILE))
print(f'Loaded {len(raw)} examples from {DATA_FILE.name}')
dataset = Dataset.from_list(raw).map(format_example, remove_columns=['messages'])
print(f'Formatted: {len(dataset)} rows')
print('
Sample (first 800 chars):')
print(dataset[0]['text'][:800])


## Cell 7 -- Train

In [ ]:
sft_config = SFTConfig(
    output_dir                  = str(OUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    save_strategy               = 'epoch',
    optim                       = 'adamw_8bit',
    dataset_text_field          = 'text',
    max_seq_length              = MAX_SEQ_LEN,
    dataset_num_proc            = 2,
    report_to                   = 'none',
)
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args          = sft_config,
)
print(f'Training: {len(dataset)} examples | {EPOCHS} epochs | effective batch={BATCH_SIZE * GRAD_ACCUM}')
trainer.train()
print('Training complete.')
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print(f'Adapter saved to {OUT_DIR}')


## Cell 8a -- Build llama.cpp (skip if already built)

In [ ]:
import subprocess, shutil, sys

CUDA_ARCH = '89'  # RTX 3090; A100=80; H100=90

llama_q = Path('/workspace/llama.cpp/build/bin/llama-quantize')
if llama_q.exists():
    print('llama.cpp already built -- skipping.')
else:
    cmds = [
        'apt-get install -y cmake build-essential 2>/dev/null',
        'cd /workspace && git clone https://github.com/ggerganov/llama.cpp || true',
        f'cd /workspace/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} && cmake --build build --config Release -j$(nproc)',
    ]
    for cmd in cmds:
        print(f'>> {cmd[:80]}')
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print('STDERR:', r.stderr[-500:])
            raise RuntimeError(f'Failed: {cmd}')
    print('llama.cpp build complete.')

LLAMA_CONVERT  = Path('/workspace/llama.cpp/convert_hf_to_gguf.py')
LLAMA_QUANTIZE = Path('/workspace/llama.cpp/build/bin/llama-quantize')
print(f'convert:  {LLAMA_CONVERT.exists()}')
print(f'quantize: {LLAMA_QUANTIZE.exists()}')


## Cell 8b -- Merge LoRA -> F16 GGUF -> Q4_K_M GGUF

In [ ]:
import os, subprocess, shutil as _shutil, sys
from pathlib import Path

# Force HF_HOME to RAM before merge -- on retrain the kernel restart loses Cell 2's
# env var and vast.ai defaults HF_HOME to /workspace, causing the 16GB BF16 download
# to land on disk and fill the 50GB volume.
os.environ['HF_HOME'] = '/dev/shm/huggingface'

# Remove stale /workspace/huggingface if it exists (leftover from a prior run where
# HF_HOME was wrong -- it's the 4-bit training cache, not needed for merge).
_stale_cache = Path('/workspace/huggingface')
if _stale_cache.exists():
    print(f'Removing stale disk cache {_stale_cache} ...')
    _shutil.rmtree(str(_stale_cache))
    subprocess.run('df -h /workspace', shell=True)

Q8_GGUF = Path('/workspace/aria-v6-q8.gguf')

# -- Step 1: Merge LoRA -> float16 safetensors ---------------------------------
if not list(GGUF_DIR.glob('*.safetensors')):
    print('Merging LoRA -> float16 ...')
    if GGUF_DIR.exists(): _shutil.rmtree(str(GGUF_DIR))
    GGUF_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained_merged(str(GGUF_DIR), tokenizer, save_method="merged_16bit")
    print(f'Merged -> {GGUF_DIR}')
else:
    print('Merged weights exist -- skipping merge.')

# -- Step 2: Convert -> q8_0 GGUF (~8GB) --------------------------------------
if not Q8_GGUF.exists():
    print('Converting to q8_0 GGUF ...')
    r = subprocess.run(
        [sys.executable, str(LLAMA_CONVERT), str(GGUF_DIR),
         '--outfile', str(Q8_GGUF), '--outtype', 'q8_0'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr[-1500:]); raise RuntimeError('q8_0 conversion failed')
    print(f'q8_0: {Q8_GGUF.stat().st_size / 1e9:.1f} GB')
    print('Removing merged safetensors to free disk ...')
    _shutil.rmtree(str(GGUF_DIR))
    print('Freed. Disk now:')
    subprocess.run('df -h /workspace', shell=True)
else:
    print(f'q8_0 exists ({Q8_GGUF.stat().st_size / 1e9:.1f} GB) -- skipping.')

## Cell 8c -- Merge + q8_0 GGUF + Q4_K_M (disk-space-aware pipeline)

Use this instead of 8b when disk is tight (&lt;20GB free).
Converts to q8_0 (~8GB) first, deletes the 15GB safetensors, then quantises to Q4_K_M.

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

# Upload whichever GGUF exists -- q8_0 if Q4_K_M not yet produced
Q8_GGUF  = Path('/workspace/aria-v2-q8.gguf')
UPLOAD_FILE = Q4KM_GGUF if Q4KM_GGUF.exists() else Q8_GGUF
if not UPLOAD_FILE.exists():
    raise FileNotFoundError('No GGUF file found -- run Cell 8b or 8c first.')

api = HfApi(token=HF_TOKEN)
try:
    api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
    print(f'Repo ready: {HF_REPO}')
except Exception as e:
    print(f'Repo: {e}')

print(f'Uploading {UPLOAD_FILE.name} ({UPLOAD_FILE.stat().st_size / 1e9:.1f} GB) ...')
api.upload_file(
    path_or_fileobj = str(UPLOAD_FILE),
    path_in_repo    = UPLOAD_FILE.name,
    repo_id         = HF_REPO,
    repo_type       = 'model',
)
print(f'Done: https://huggingface.co/{HF_REPO}')
print(f'File uploaded: {UPLOAD_FILE.name}')


## Cell 9 -- Upload GGUF to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
try:
    api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
    print(f'Repo ready: {HF_REPO}')
except Exception as e:
    print(f'Repo: {e}')

print(f'Uploading {Q4KM_GGUF.name} ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB) ...')
api.upload_file(
    path_or_fileobj = str(Q4KM_GGUF),
    path_in_repo    = Q4KM_GGUF.name,
    repo_id         = HF_REPO,
    repo_type       = 'model',
)
print(f'Done: https://huggingface.co/{HF_REPO}')


## Cell 10 -- Install Ollama (skip if already installed)

In [ ]:
import subprocess, shutil

if shutil.which("ollama"):
    print("Ollama already installed:", shutil.which("ollama"))
else:
    print("Installing Ollama ...")
    r = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    print(r.stdout[-500:])
    if r.returncode != 0:
        print(r.stderr[-500:])
        raise RuntimeError("Ollama install failed")
    print("Ollama installed.")

r = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
print(r.stdout.strip())


## Cell 11 -- Write Modelfile & Register with Ollama

Gemma 4 uses `<start_of_turn>` / `<end_of_turn>` chat tokens.

> **VERIFY:** Confirm the TEMPLATE syntax against the Ollama Gemma library page
> before running: https://ollama.com/library/gemma3 (use equivalent for Gemma 4)


In [ ]:
import subprocess
from pathlib import Path

Q8_GGUF = Path('/workspace/aria-v6-q8.gguf')

if not Q8_GGUF.exists():
    raise FileNotFoundError(f'{Q8_GGUF} not found -- run Cell 8b first.')

print(f'Using GGUF: {Q8_GGUF}  ({Q8_GGUF.stat().st_size / 1e9:.1f} GB)')

SYSTEM_PROMPT = (
    'You are an AML (Anti-Money Laundering) analytics AI assistant. '
    'You analyze false positive/false negative trade-offs in AML alert thresholds, '
    'perform customer behavioral segmentation, and interpret clustering results. '
    'Use the available tools to retrieve data, then provide clear, analytical insights. '
    'IMPORTANT: You MUST respond entirely in English. '
    'Be concise and reference specific numbers when interpreting results.'
)

lines = [
    f'FROM /workspace/aria-v6-q8.gguf',
    '',
    'PARAMETER num_ctx 8192',
    'PARAMETER temperature 0.1',
    'PARAMETER top_p 0.9',
    'PARAMETER stop <turn|>',
    'PARAMETER stop <eos>',
    '',
    'TEMPLATE """',
    '{{ if .System }}<|turn>user',
    '{{ .System }}<turn|>',
    '{{ end }}',
    '{{ range .Messages }}',
    '{{ if eq .Role "user" }}<|turn>user',
    '{{ .Content }}<turn|>',
    '<|turn>model',
    '{{ else if eq .Role "assistant" }}{{ .Content }}<turn|>',
    '{{ else if eq .Role "tool" }}<|turn>tool',
    '{{ .Content }}<turn|>',
    '<|turn>model',
    '{{ end }}',
    '{{ end }}',
    '"""',
    '',
    f'SYSTEM "{SYSTEM_PROMPT}"',
]

modelfile_content = chr(10).join(lines)
modelfile_path = Path('/tmp/Modelfile.aml')
modelfile_path.write_text(modelfile_content, encoding='utf-8')
print('--- Modelfile ---')
print(modelfile_content)

## Cell 12 -- Start Ollama Server

Binds to 0.0.0.0:11434. Expose via vast.ai port forwarding or Cloudflare tunnel.


In [ ]:
import time

subprocess.run("pkill -f 'ollama serve' || true", shell=True)
time.sleep(2)
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
proc = subprocess.Popen(['ollama', 'serve'], env=env,
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print(f'Ollama server PID: {proc.pid}')
r = subprocess.run(['curl', '-s', 'http://localhost:11434/api/tags'],
                   capture_output=True, text=True)
print('Health check:', r.stdout[:200])


In [ ]:
print(f'Registering {OLLAMA_MODEL_NAME} ...')
result = subprocess.run(
    ['ollama', 'create', OLLAMA_MODEL_NAME, '-f', str(modelfile_path)],
    capture_output=True, text=True
)
print(result.stdout[-500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('ollama create failed')
r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print('Registered models:')
print(r.stdout)


## Cell 14 -- Smoke Test

In [ ]:
import urllib.request, json as _json

payload = _json.dumps({
    'model': OLLAMA_MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Show FP/FN trade-off for Business customers by monthly transaction amount'}],
    'stream': False,
}).encode()
req = urllib.request.Request(
    'http://localhost:11434/api/chat',
    data=payload,
    headers={'Content-Type': 'application/json'},
)
with urllib.request.urlopen(req, timeout=120) as resp:
    result = _json.loads(resp.read())
print('Response:')
print(result['message']['content'][:800])


## Cell 15 -- Connect Local Dash App

On your **local Windows machine**:

```powershell
$env:OLLAMA_BASE_URL = "http://<vast-ai-ip>:11434/v1"
$env:OLLAMA_MODEL    = "aria-v2"
.venv\Scripts\python.exe application.py
```

Or use a Cloudflare tunnel on the vast.ai instance:
```bash
cloudflared tunnel --url http://localhost:11434
```
